# Step 1 – Exploring and downloading possible Biomass stacks over Antarctica

This code is adapted from the Examples_and_Data/Coding/Biomass/Data Access/Biomass Data Access Example.ipynb

### 1. Packages and Libraries

In [ ]:
# %pip install aiohttp requests netCDF4 xarray folium shapely pillow

from shapely.geometry import Polygon
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os

import polsar_utils as pu
from BPS_inputs_download import get_stack
from pathlib import Path

fiona.supported_drivers["KML"] = "rw"  # Enable kml driver

### 2. Query the ESA MAAP STAC catalog

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

#### 2.1. Find the collection

In [ ]:
# find all Biomass collections
results = catalog.collection_search(
    filter="platform='Biomass'",
    datetime="2025-12-01T00:00:00.000Z/..",
)
print(f"{results.matched()} collections found.")

data = results.collection_list_as_dict()
df = pd.json_normalize(data, record_path=["collections"]).sort_values(
    by="title", ignore_index=True
)
df[["id", "title", "extent.temporal.interval"]]

#### 2.2. Select L1C products and narrow down the search

We are selecting L1C BIOMASS products as they contain stacks of scenes suitable for interferometry and tomography. The available querables are available here: [Queryables for L1c products](https://catalog.maap.eo.esa.int/catalogue/collections/BiomassLevel1c/queryables).

In [ ]:
search = catalog.search(
    collections = ["BiomassLevel1c"],
    filter = "product:type='S1_STA__1S' or product:type='S2_STA__1S' or product:type='S3_STA__1S'",
    datetime = ['2025-01-01T00:00:00Z','2026-10-13T23:59:59Z'],
    # bbox = [-75.1451, -71.5351, -69.0196, -69.4491],  # Wilkins Ice Shelf, Antarctica
    # bbox = [94.6319, -67.9541, 103.7777, -63.7139], # Denman Glacier
    #bbox = [-107.7404, -75.5030, -98.5351, -73.3782], # PIG
    # bbox = [-157.0226, -84.6933, 163.2812, -75.9024], # Ross Ice shelf
    # bbox = [-126.0448, -57.8721, 128.0156, -53.8483], # whole Antarctica
    # bbox = [7.272949,-71.566100,13.996582,-69.660815], #Nivilsen
    bbox = [ -75.0, -73.5, -66.5, -69.0],
    method = "GET",
    max_items = 320,  # change as needed
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

#### 2.3. Visualise the returned IDs into MAAP

In [ ]:
features = []
for it in items:
    if it.geometry is None:
        raise ValueError(f"Item {it.id} has no geometry (footprint).")
    props = it.properties or {}
    features.append({
        "type": "Feature",
        "geometry": it.geometry,
        "properties": {
            "product_id": it.id,
            "stack_id": props.get("eofeos:stack_id"),  
            "start_datetime": props.get("start_datetime"),
            "end_datetime": props.get("end_datetime"),
        },
    })

gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
gdf = gdf.reset_index()
gdf['stack_number'] = gdf.groupby('stack_id').ngroup() + 1
gdf['products_in_stack'] = gdf.groupby('stack_id')['product_id'].transform('count')
gdf = gdf[["index", "product_id", "stack_id", "stack_number", "products_in_stack", "start_datetime", "end_datetime", "geometry"]]
gdf.head(len(items))

In [ ]:
## Ensure both are WGS84 for folium
gdf = gdf.to_crs("EPSG:4326")

# Center/zoom using bounds
bounds = gpd.GeoSeries(list(gdf.geometry), crs="EPSG:4326")
minx, miny, maxx, maxy = bounds.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=3)

def style(_):
    return {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.10}  

GeoJson(
    gdf.__geo_interface__,
    name=f"Products ({len(gdf)})",
    tooltip=folium.GeoJsonTooltip(fields=["product_id", "index", "stack_id", "stack_number"]),
    style_function=style,
).add_to(m)


folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m

#### 2.4. Choose your stack

By hovering over the map above, you can see the product ID, the index in the geodatabase and the stack ID and stack number (stack number is a unique number for each stack_id, as stack_id is a very long string).

Pick the stack you want to download and insert the stack_number in the line below.

In [ ]:
CHOSEN_STACK_NUMBER = 45 #pick stack number

In [ ]:
gdf_subset = gdf[gdf['stack_number'] == CHOSEN_STACK_NUMBER].copy()

# Ensure both are WGS84 for folium
gdf_subset = gdf_subset.to_crs("EPSG:4326")

# Center/zoom using bounds of the filtered stack
bounds = gpd.GeoSeries(list(gdf_subset.geometry), crs="EPSG:4326")
minx, miny, maxx, maxy = bounds.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=6)

def style(_):
    return {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.20}  

GeoJson(
    gdf_subset.__geo_interface__,
    name=f"Stack {CHOSEN_STACK_NUMBER} ({len(gdf_subset)} products)",
    tooltip=folium.GeoJsonTooltip(
        fields=["product_id", "index", "stack_id", "stack_number", "products_in_stack"],
        aliases=["Product ID:", "Index:", "Stack ID:", "Stack Number:", "Total in Stack:"]
    ),
    style_function=style,
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m

#### 3.4. Download your stack

Download the stack to the chosen DOWNLOAD_FOLDER.

In [ ]:
DOWNLOAD_FOLDER = Path("biomass_inputs")

In [ ]:
chosen_stack_id = gdf[gdf['stack_number'] == CHOSEN_STACK_NUMBER]['stack_id'].iloc[0]
print(chosen_stack_id)

In [ ]:
from collections import defaultdict
stacks = defaultdict(list)
for idx, item in enumerate(items):
    stack_id = (item.properties or {}).get("eofeos:stack_id")
    if stack_id:
        stacks[stack_id].append(idx)

# Use your chosen stack ID string directly (e.g., from your gdf subset)
chosen_stack_id = gdf[gdf['stack_number'] == CHOSEN_STACK_NUMBER]['stack_id'].iloc[0]
if chosen_stack_id not in stacks:
    raise ValueError(f"Stack ID {chosen_stack_id} not found in the items list.")

indices = stacks[chosen_stack_id]

reference_id = items[indices[0]].id

safe_stack_name = chosen_stack_id.replace(":", "_").replace("/", "_")
stack_folder = DOWNLOAD_FOLDER / safe_stack_name
stack_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
print(f"\nDownloading Stack {CHOSEN_STACK_NUMBER} | Stack ID: {stack_id}")
print(f"Catalog indices in stack: {indices}")
print(f"Reference L1C ID: {reference_id}")
print(f"Saving to: {stack_folder.resolve()}")

# Perform the download
get_stack(reference_id, str(stack_folder))
print(f"Completed downloading stack {stack_id}.")

#### 3.5. Inspect

In [ ]:
stacks = pu.find_stacks(DOWNLOAD_FOLDER)
pu.print_overview(stacks)